# CellProfiler morphology analysis

**Template** — edit the *Configuration* cell and run all cells.

**Pre-requisite:** run `Results/classify_objects.py` first if any condition contains a second
cell type that must be separated. That script writes `predictions.csv` to the Results folder.
If all wells are pure NSC, set `PREDICTIONS_CSV = None`.

Input: CellProfiler CellposeObjects CSV + SeedObjects CSV (skeleton) + `plate_map.csv`.
Output: QC chart, paired boxplots per metric group, Z-score heatmap, statistics vs Control.

In [ ]:
import pathlib, sys
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats as scipy_stats

# repo root — adjust if notebook is not in notebooks/<subdir>/
REPO_ROOT = pathlib.Path().resolve().parents[1]
sys.path.insert(0, str(REPO_ROOT))
import incucyte as ic

## Configuration

In [ ]:
# ── EDIT THESE ──────────────────────────────────────────────────────────────
EXP_DIR        = pathlib.Path(r"TODO\path\to\experiment")

CELLPOSE_CSV   = EXP_DIR / "Results" / "measurements" / "MyCellProfilerExpt_CellposeObjects.csv"
SEED_CSV       = EXP_DIR / "Results" / "measurements" / "MyCellProfilerExpt_SeedObjects.csv"
PLATE_MAP_CSV  = EXP_DIR / "plate_map.csv"
PREDICTIONS_CSV = EXP_DIR / "Results" / "predictions.csv"  # set to None if not needed
OUTPUT_DIR     = EXP_DIR / "Results"                        # for saved PDFs and stats TSV

# Condition display order (left → right in boxplots, top → bottom in heatmap)
CONDITIONS_ORDER = ["Control", "TODO"]  # list all conditions
CONTROL = "Control"                      # baseline for pairwise statistics

# CellposeObjects columns to use as shape metrics
SHAPE_METRICS = [
    "AreaShape_Area",
    "AreaShape_Perimeter",
    "AreaShape_FormFactor",
    "AreaShape_Solidity",
    "AreaShape_MajorAxisLength",
    "AreaShape_MinorAxisLength",
    "AreaShape_Eccentricity",
    "AreaShape_EquivalentDiameter",
]

# SeedObjects columns to use as skeleton metrics
SKEL_METRICS = [
    "ObjectSkeleton_NumberBranchEnds_SkeletonImage",
    "ObjectSkeleton_NumberNonTrunkBranches_SkeletonImage",
    "ObjectSkeleton_NumberTrunks_SkeletonImage",
    "ObjectSkeleton_TotalObjectSkeletonLength_SkeletonImage",
]

# Metric grouping and colours for the heatmap column annotation
METRIC_CLASSES = {
    "AreaShape_Area":               "Shape — Size",
    "AreaShape_Perimeter":          "Shape — Size",
    "AreaShape_MajorAxisLength":    "Shape — Size",
    "AreaShape_MinorAxisLength":    "Shape — Size",
    "AreaShape_EquivalentDiameter": "Shape — Size",
    "AreaShape_FormFactor":         "Shape — Form",
    "AreaShape_Solidity":           "Shape — Form",
    "AreaShape_Eccentricity":       "Shape — Form",
    "ObjectSkeleton_NumberBranchEnds_SkeletonImage":          "Skeleton",
    "ObjectSkeleton_NumberNonTrunkBranches_SkeletonImage":    "Skeleton",
    "ObjectSkeleton_NumberTrunks_SkeletonImage":              "Skeleton",
    "ObjectSkeleton_TotalObjectSkeletonLength_SkeletonImage": "Skeleton",
}
CLASS_COLORS = {
    "Shape — Size": "#574DE7",
    "Shape — Form": "#C736AF",
    "Skeleton":      "#2ec93b",
}

# Heatmap row annotation colours — one entry per condition
CONDITION_COLORS = {"Control": "#4878CF", "TODO": "#D65F5F"}

# Short column labels for the heatmap (optional — set to None to use full names)
SHORT_NAMES = {
    "AreaShape_Area":               "Area",
    "AreaShape_Perimeter":          "Perimeter",
    "AreaShape_FormFactor":         "Form factor",
    "AreaShape_Solidity":           "Solidity",
    "AreaShape_MajorAxisLength":    "Major axis",
    "AreaShape_MinorAxisLength":    "Minor axis",
    "AreaShape_Eccentricity":       "Eccentricity",
    "AreaShape_EquivalentDiameter": "Equiv. diam.",
    "ObjectSkeleton_NumberBranchEnds_SkeletonImage":          "Branch ends",
    "ObjectSkeleton_NumberNonTrunkBranches_SkeletonImage":    "Branches",
    "ObjectSkeleton_NumberTrunks_SkeletonImage":              "Trunks",
    "ObjectSkeleton_TotalObjectSkeletonLength_SkeletonImage": "Skel. length",
}
# ────────────────────────────────────────────────────────────────────────────

## 1. Load data

In [ ]:
# Load CellposeObjects (shape + texture) and join skeleton from SeedObjects
df = ic.load_cellprofiler(CELLPOSE_CSV)
seed = pd.read_csv(SEED_CSV, usecols=["ImageNumber", "ObjectNumber"] + SKEL_METRICS,
                   low_memory=False)
df = df.merge(seed, on=["ImageNumber", "ObjectNumber"], how="left")

# Join predicted labels; objects not in predictions are pure NSC wells
if PREDICTIONS_CSV is not None:
    preds = pd.read_csv(PREDICTIONS_CSV,
                        usecols=["FileName_PC", "ObjectNumber", "predicted_label"])
    df = df.merge(preds, on=["FileName_PC", "ObjectNumber"], how="left")
df["predicted_label"] = df.get("predicted_label", pd.Series("NSC", index=df.index)).fillna("NSC")

# Keep only NSC-classified objects
df = df[df["predicted_label"] == "NSC"].copy()

# Merge plate map
plate_map = ic.load_plate_map(PLATE_MAP_CSV)
df = ic.merge_plate_map(df, plate_map)

print(f"{len(df):,} NSC objects  |  {df['Well'].nunique()} wells")
print("Counts per condition:", df.groupby("condition").size().to_dict())
df.head(3)

## 2. QC — object counts per well

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
ic.qc_counts(df, groupby="Well", ax=ax)
ax.set_title("NSC object count per well (QC)")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "QC_object_counts.pdf", format="pdf", bbox_inches="tight")

## 3. Aggregate

Two-step: per-object → per-well mean → per-biological-replicate (sample) mean.
Skeleton: filter to objects with at least one detected skeleton branch before aggregating.

In [ ]:
# Shape
df_well_shape = ic.to_well_means(df, metrics=SHAPE_METRICS)
df_bio_shape = df_well_shape.groupby(["sample", "condition"])[SHAPE_METRICS].mean().reset_index()

print(f"Shape observations: {len(df_bio_shape)}")
print(df_bio_shape.groupby("condition").size().to_dict())
df_bio_shape.head(4)

In [ ]:
# Skeleton: keep objects that have any skeleton measurement
df_skel = ic.filter_not_null(df, cols=[SKEL_METRICS[0]])
print(f"Objects with skeleton data: {len(df_skel):,} / {len(df):,} ({100*len(df_skel)/len(df):.1f}%)")

df_well_skel = ic.to_well_means(df_skel, metrics=SKEL_METRICS)
df_bio_skel = df_well_skel.groupby(["sample", "condition"])[SKEL_METRICS].mean().reset_index()

print(f"Skeleton observations: {len(df_bio_skel)}")
df_bio_skel.head(4)

## 4. Paired boxplots — shape metrics

Each point is one biological replicate (sample); lines connect the same sample across conditions.

In [ ]:
ncols = 4
nrows = -(-len(SHAPE_METRICS) // ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 5 * nrows))
axes = axes.flatten()
for i, metric in enumerate(SHAPE_METRICS):
    ic.paired_boxplot(df_bio_shape, y=metric, condition_col="condition",
                      pair_col="sample", order=CONDITIONS_ORDER, ax=axes[i])
    axes[i].set_title(metric, fontsize=9)
    axes[i].set_xlabel("")
for j in range(len(SHAPE_METRICS), len(axes)):
    axes[j].set_visible(False)
fig.suptitle("Shape metrics — NSC", fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "Shape_metrics.pdf", format="pdf", bbox_inches="tight")

## 5. Paired boxplots — skeleton metrics

In [ ]:
ncols = 4
nrows = -(-len(SKEL_METRICS) // ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 5 * nrows))
axes = axes.flatten()
for i, metric in enumerate(SKEL_METRICS):
    ic.paired_boxplot(df_bio_skel, y=metric, condition_col="condition",
                      pair_col="sample", order=CONDITIONS_ORDER, ax=axes[i])
    axes[i].set_title(metric, fontsize=9)
    axes[i].set_xlabel("")
for j in range(len(SKEL_METRICS), len(axes)):
    axes[j].set_visible(False)
fig.suptitle("Skeleton metrics — NSC with detected skeleton", fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "Skeleton_metrics.pdf", format="pdf", bbox_inches="tight")

## 6. Z-score heatmap

Rows = (condition, sample), sorted by CONDITIONS_ORDER.  
Columns = all metrics grouped by class; colour bar shows metric class.  
Values are Z-scored across rows (per metric, mean 0 ± 1 SD).

In [ ]:
df_bio_all = df_bio_shape.merge(df_bio_skel, on=["sample", "condition"], how="outer")

g = ic.metric_heatmap(
    df_bio_all,
    metrics=SHAPE_METRICS + SKEL_METRICS,
    metric_classes=METRIC_CLASSES,
    class_colors=CLASS_COLORS,
    row_col="condition",
    row_order=CONDITIONS_ORDER,
    sample_col="sample",
    condition_colors=CONDITION_COLORS,
    short_names=SHORT_NAMES,
    col_cluster=True,
    title="Mean metric profiles per sample × condition (Z-scored)",
)
g.fig.savefig(OUTPUT_DIR / "metric_profiles_heatmap.pdf", format="pdf", bbox_inches="tight")
plt.show()

## 7. Statistics — each condition vs Control

Shapiro–Wilk test on paired differences → paired t-test (normal) or Wilcoxon (non-normal).  
Bonferroni correction for the number of non-control conditions tested.

In [ ]:
def _stats_vs_control(df_bio, metric, conditions_order, control, pair_col="sample"):
    ctrl = df_bio[df_bio["condition"] == control].set_index(pair_col)[metric]
    rows = []
    for cond in conditions_order:
        if cond == control:
            continue
        grp = df_bio[df_bio["condition"] == cond].set_index(pair_col)[metric]
        a, b = ctrl.align(grp, join="inner")
        diff = (b - a).dropna()
        n = len(diff)
        row = {"comparison": f"{cond} vs {control}", "metric": metric, "n": n}
        if n < 3:
            row.update(test="insufficient n", statistic=None, p_raw=None, shapiro_p=None)
        else:
            _, p_sw = scipy_stats.shapiro(diff)
            if p_sw > 0.05:
                stat, p = scipy_stats.ttest_rel(a.values, b.values)
                row["test"] = "paired t-test"
            else:
                stat, p = scipy_stats.wilcoxon(a.values, b.values)
                row["test"] = "Wilcoxon"
            row.update(statistic=round(float(stat), 4),
                       p_raw=round(float(p), 4),
                       shapiro_p=round(float(p_sw), 4))
        rows.append(row)
    return rows


n_comparisons = len(CONDITIONS_ORDER) - 1  # number of non-control conditions
results = []
for m in SHAPE_METRICS:
    results.extend(_stats_vs_control(df_bio_shape, m, CONDITIONS_ORDER, CONTROL))
for m in SKEL_METRICS:
    results.extend(_stats_vs_control(df_bio_skel, m, CONDITIONS_ORDER, CONTROL))

df_stats = pd.DataFrame(results)
df_stats["p_bonf"] = (df_stats["p_raw"] * n_comparisons).clip(upper=1.0)
df_stats["significant"] = df_stats["p_bonf"] < 0.05

print(df_stats.to_string(index=False))
df_stats.to_csv(OUTPUT_DIR / "stats_vs_control.tsv", sep="\t", index=False)